# German Cities Data Pipeline

Small ETL project that pulls together demographic, geographic, weather, airport, and flight data for a set of German cities (Munich, Stuttgart, Düsseldorf) and stores it in a MySQL database.

**Before running this notebook:**
1. Create the database schema first — run `schema.sql` against your MySQL instance.
2. Copy `.env.example` to `.env` and fill in your own credentials/API keys.
3. Install dependencies from `requirements.txt`.


## Step 0: Setup

### Cell 0.1: Environment Setup & Database Connection

In [ ]:
import os
from datetime import datetime
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine

# Load environment variables
load_dotenv()
schema = os.getenv("DB_SCHEMA")
host = os.getenv("DB_HOST")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
port = int(os.getenv("DB_PORT", 3306))

# Establish database connection engine
connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"
engine = create_engine(connection_string)

# Define target cities
target_cities = ["Munich", "Stuttgart", "Düsseldorf"]
print("Environment loaded and engine connected successfully.")


### Cell 0.2: Helper Function — Safe Insert (`INSERT IGNORE`)

`pandas.to_sql()` doesn't support `INSERT IGNORE` out of the box — the `method` argument only accepts `None`, `"multi"`, or a callable. This helper wraps SQLAlchemy's MySQL insert with an `IGNORE` prefix so re-running the notebook against the same database won't raise duplicate-key errors.

Note: `INSERT IGNORE` only has something to ignore where the target table has a `PRIMARY KEY` or `UNIQUE` constraint — see `schema.sql` for the constraints this relies on.

In [ ]:
from sqlalchemy.dialects.mysql import insert as mysql_insert

def insert_ignore(table, conn, keys, data_iter):
    """Custom to_sql `method` that translates to MySQL's INSERT IGNORE."""
    data = [dict(zip(keys, row)) for row in data_iter]
    stmt = mysql_insert(table.table).values(data).prefix_with("IGNORE")
    conn.execute(stmt)


## Step 1: Wikipedia Web Scraping (Demographic & Geographic Data)

### Cell 1.1: Filter New Cities & Scrape Wikipedia

In [ ]:
from bs4 import BeautifulSoup
from lat_lon_parser import parse
import requests

# Only process cities not already in the database
existing_cities = pd.read_sql("SELECT city_name FROM cities", con=engine)["city_name"].tolist()
cities_to_process = [c for c in target_cities if c not in existing_cities]

scraped_data = []
current_year = datetime.today().year

if not cities_to_process:
    print("All target cities already exist — nothing to scrape.")
else:
    print(f"New cities to process: {cities_to_process}")

for city in cities_to_process:
    url = f"https://www.wikipedia.org/wiki/{city}"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
            " (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36"
        )
    }

    # Fetch webpage content
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        continue

    city_soup = BeautifulSoup(response.content, "html.parser")

    try:
        # Extract geographic coordinates and population data
        city_latitude = city_soup.find(class_="latitude").get_text()
        city_longitude = city_soup.find(class_="longitude").get_text()
        city_population = (
            city_soup.find(string="Population").find_next("td").get_text()
        )
        city_population_clean = int(
            city_population.split("[")[0].replace(",", "").strip()
        )

        scraped_data.append({
            "city_name": city,
            "country_code": "DEU",
            "latitude": parse(city_latitude),
            "longitude": parse(city_longitude),
            "population": city_population_clean,
            "timestamp_population": current_year,
        })
    except AttributeError as e:
        print(f"Error parsing data for {city}: {e}")

# Create Master DataFrame
df_scraped_master = pd.DataFrame(scraped_data)
display(df_scraped_master)


### Cell 1.2: Load into MySQL (Cities, Populations, Geo)

In [ ]:
if df_scraped_master.empty:
    print("Step 1 skipped: no new cities to insert.")
else:
    # 1. Insert master city records into 'cities' table
    df_new_cities = df_scraped_master[["city_name", "country_code"]]
    df_new_cities.to_sql(
        "cities", con=engine, if_exists="append", index=False, method=insert_ignore
    )

    # 2. Fetch generated city_ids from SQL and merge back with data
    df_cities_from_sql = pd.read_sql(
        "SELECT city_id, city_name FROM cities", con=engine
    )
    df_merged = pd.merge(df_scraped_master, df_cities_from_sql, on="city_name")

    # 3. Insert population records into 'populations' table
    df_populations = df_merged[["city_id", "population", "timestamp_population"]]
    df_populations.to_sql(
        "populations", con=engine, if_exists="append", index=False, method=insert_ignore
    )

    # 4. Insert geographic records into 'geo' table
    df_geo = df_merged[["city_id", "latitude", "longitude"]]
    df_geo.to_sql(
        "geo", con=engine, if_exists="append", index=False, method=insert_ignore
    )

    print(f"Step 1 completed: {len(df_new_cities)} new city record(s) added.")


## Step 2: Weather Data Extraction

### Cell 2.1: Fetching City IDs from Database

In [ ]:
# Retrieve city IDs and names from MySQL
df_cities = pd.read_sql("SELECT city_id, city_name FROM cities", con=engine)
display(df_cities)


### Cell 2.2: API Extraction & DataFrame Transformation

In [ ]:
api_key = os.getenv("OPENWEATHER_API_KEY")
forecast_records = []

# Loop through cities to fetch weather forecasts from OpenWeatherMap API
for index, row in df_cities.iterrows():
    city_id = row["city_id"]
    city_name = row["city_name"]

    url = f"https://api.openweathermap.org/data/2.5/forecast?q={city_name}&appid={api_key}&units=metric"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        for entry in data["list"]:
            forecast_records.append({
                "city_id": city_id,
                "forecast_time": entry["dt_txt"],
                "outlook": entry["weather"][0]["description"],
                "temperature": entry["main"]["temp"],
                "feels_like": entry["main"]["feels_like"],
                "wind_speed": entry["wind"]["speed"],
                "rain_prob": entry.get("pop", 0.0),
            })
    else:
        print(f"Failed to fetch weather for {city_name}: {response.status_code}")

# Build Forecast DataFrame
df_forecast = pd.DataFrame(forecast_records)
display(df_forecast.head())


### Cell 2.3: Loading Weather Data to MySQL

In [ ]:
if not df_forecast.empty:
    df_forecast.to_sql(
        "weathers", con=engine, if_exists="append", index=False, method=insert_ignore
    )
    print("Step 2 completed: Weather forecast data loaded successfully.")
else:
    print("No weather records were collected.")


## Step 3: Airports Extraction & Bridge Mapping

### Cell 3.1: Retrieving City Coordinates

In [ ]:
# Join cities and geo tables to get coordinates
df_city_coords = pd.read_sql(
    """
    SELECT c.city_id, c.city_name, g.latitude, g.longitude
    FROM cities c
    JOIN geo g ON c.city_id = g.city_id
    """,
    con=engine,
)
display(df_city_coords)


### Cell 3.2: API Request & DataFrame Creation

In [ ]:
RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")
airport_records = []
city_airport_mappings = []

headers = {
    "x-rapidapi-key": RAPIDAPI_KEY,
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
    "Content-Type": "application/json",
}

# Fetch nearby airports using AeroDataBox location endpoint
for _, row in df_city_coords.iterrows():
    city_id = row["city_id"]
    lat = row["latitude"]
    lon = row["longitude"]

    url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
    querystring = {
        "lat": lat,
        "lon": lon,
        "radiusKm": "50",
        "limit": "10",
        "withFlightInfoOnly": "true",
    }

    response = requests.get(url, headers=headers, params=querystring)
    if response.status_code == 200:
        items = response.json().get("items", [])
        for item in items:
            icao = item.get("icao")
            name = item.get("name")
            if icao:
                airport_records.append({"airport_icao": icao, "airport_name": name})
                city_airport_mappings.append({"city_id": city_id, "airport_icao": icao})

# Create unique DataFrames for airports and bridge mapping
df_airports = pd.DataFrame(airport_records).drop_duplicates(subset=["airport_icao"])
df_bridge = pd.DataFrame(city_airport_mappings).drop_duplicates()

print(f"Found {len(df_airports)} unique airports.")
display(df_airports)


### Cell 3.3: Loading Airports & Bridge Data to MySQL

In [ ]:
# Load data into 'airports' and 'cities_airports' tables
if not df_airports.empty:
    df_airports.to_sql(
        "airports", con=engine, if_exists="append", index=False, method=insert_ignore
    )
if not df_bridge.empty:
    df_bridge.to_sql(
        "cities_airports", con=engine, if_exists="append", index=False, method=insert_ignore
    )
print("Step 3 completed: Airports and bridge mappings loaded.")


## Step 4: Flight Data Extraction

### Cell 4.1: Fetching Airport ICAO Codes

In [ ]:
# Query distinct airport ICAOs from database
query = "SELECT DISTINCT airport_icao FROM airports WHERE airport_icao IS NOT NULL"
airport_icaos = pd.read_sql(query, con=engine)["airport_icao"].tolist()
print("Airports to fetch flights for:", airport_icaos)


### Cell 4.2: API Request & Flights Processing

In [ ]:
import time
from datetime import timedelta, timezone

RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")
BASE_URL = "https://aerodatabox.p.rapidapi.com/flights/airports/icao"
HEADERS = {
    "Content-Type": "application/json",
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
    "x-rapidapi-key": RAPIDAPI_KEY,
}

# Define UTC time window for the next 12 hours
now_utc = datetime.now(timezone.utc)
time_from = now_utc.strftime("%Y-%m-%dT%H:%M")
time_to = (now_utc + timedelta(hours=12)).strftime("%Y-%m-%dT%H:%M")

flight_records = []

# Fetch arrival flights for each airport code
for icao in airport_icaos:
    url = f"{BASE_URL}/{icao}/{time_from}/{time_to}"
    querystring = {
        "withLeg": "true",
        "direction": "Arrival",
        "withCancelled": "true",
        "withCodeshared": "true",
        "withCargo": "true",
        "withPrivate": "true",
        "withLocation": "false",
    }

    response = requests.get(url, headers=HEADERS, params=querystring)
    if response.status_code == 200:
        arrivals = response.json().get("arrivals", [])
        for flight in arrivals:
            flight_num = flight.get("number")
            departure_icao = flight.get("departure", {}).get("airport", {}).get("icao")
            arrival_time = flight.get("arrival", {}).get("scheduledTime", {}).get("utc")

            if not flight_num or not arrival_time:
                continue

            # Clean datetime format for SQL insertion
            cleaned_time = arrival_time.replace("Z", "").replace("T", " ")
            flight_records.append({
                "flight_num": flight_num,
                "departure_icao": departure_icao,
                "arrival_icao": icao,
                "arrival_time": cleaned_time,
            })
    time.sleep(1)  # Rate limit handling

# Build Flights DataFrame
df_flights = pd.DataFrame(flight_records).drop_duplicates()
print(f"Total flights collected: {len(df_flights)}")
display(df_flights.head())


### Cell 4.3: Loading Flights Data into MySQL

In [ ]:
# Insert flight records into 'flights' table, skipping duplicates automatically
if not df_flights.empty:
    df_flights.to_sql(
        "flights", con=engine, if_exists="append", index=False, method=insert_ignore
    )
    print(f"Step 4 completed: fetched {len(df_flights)} flight record(s) (duplicates skipped automatically).")
else:
    print("No flight records were collected.")
